# 📊 Analisis Korelasi Beban Akademik terhadap Stabilitas Emosional Mahasiswa

| | |
|---|---|
| **Mata Kuliah** | Sampling dan Survey |
| **Dosen Pengampu** | Dr. Andi Pujo Rahadi, S.T., M.Sc. |
| **Program Studi** | S1 Sains Data — Semester 2 |
| **Institusi** | Institut Teknologi Sains Bandung (ITSB) |
| **Tahun** | 2025 |

---

### 👥 Anggota Kelompok 1

| No. | Nama | NIM |
|:---:|---|:---:|
| 1 | Zidhan Alfarezi Afdi | 52250049 |
| 2 | Den Yuan Frasseka | 52250050 |
| 3 | Boma Satrio Wicaksono Dewantoro | 52250061 |
| 4 | Naila Syahrani Putri | 52250070 |

---

### 🎯 Tujuan Penelitian
Mengetahui apakah terdapat hubungan (korelasi) yang signifikan antara **tingkat beban akademik** yang dirasakan mahasiswa dengan **stabilitas emosional** mereka, dibuktikan secara statistik menggunakan data survei dari 50 responden mahasiswa ITSB.

### 📋 Tentang Dataset
- **Sumber data:** Google Form (survei primer)
- **Jumlah responden:** 50 mahasiswa
- **Jumlah pertanyaan:** 18 pertanyaan (4 bagian: A–D)
- **Skala pengukuran:** Likert 1–5 (untuk indikator korelasi)
- **Repository dataset:** [GitHub — bomass1116](https://github.com/bomass1116)

### 📌 Alur Analisis
> **1. Import Library** → **2. Load Data** → **3. Definisi Kolom** → **4. Profil Responden** → **5–7. Analisis Deskriptif** → **8. Distribusi Data** → **9. Uji Normalitas** → **10. Uji Korelasi** → **11. Perbandingan Gender** → **12. Kesimpulan**



## 1. Persiapan: Import Library


In [ ]:
# ======================================================
# Import semua library yang aku butuhkan buat analisis.
# Pandas buat olah data, matplotlib & seaborn buat grafik,
# numpy buat hitung-hitungan, scipy buat uji statistik.
# ======================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Atur styling global — semua grafik konsisten ──────────────────────────
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi'        : 120,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 11,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'font.family'       : 'DejaVu Sans',
    'legend.fontsize'   : 10,
    'xtick.labelsize'   : 10,
    'ytick.labelsize'   : 10,
})

print('✅ Semua library berhasil diimport!')
print('✅ Global styling diatur — semua grafik akan tampil konsisten.')


## 2. Memuat Dataset

> Dataset diambil langsung dari GitHub — tidak perlu upload file manual setiap kali buka Colab.
> Pastikan terhubung ke internet saat menjalankan notebook ini.


In [ ]:
# ======================================================
# Aku load dataset langsung dari GitHub tanpa perlu
# upload manual setiap kali buka notebook ini di Colab.
# ⚠️  File ini pakai separator TITIK KOMA (;) — bukan
#     koma biasa, jadi wajib tambahkan sep=';'
# ======================================================

# URL dataset di GitHub (raw)
GITHUB_URL = (
    "https://raw.githubusercontent.com/bomass1116/Dataset_survey/"
    "523fa2def962cac4651489be1c47226c6bdc81d2/"
    "Dataset_Korelasi_Beban_Akademik_terhadap_Stabilitas_Emosional_%20(Jawaban).csv"
)

# Load dataset
# sep=';'          → karena file pakai titik koma sebagai pemisah kolom
# encoding='utf-8-sig' → mengatasi karakter BOM (\ufeff) di awal file
df = pd.read_csv(GITHUB_URL, sep=';', encoding='utf-8-sig')

print('✅ Dataset berhasil dimuat dari GitHub!')
print(f'   Jumlah responden  : {len(df)} orang')
print(f'   Jumlah kolom      : {len(df.columns)} kolom')
print(f'   Separator file    : titik koma (;)')
print(f'   Data kosong       : {df.isnull().sum().sum()} sel')
print()
print('Daftar kolom:')
for i, col in enumerate(df.columns, 1):
    print(f'  [{i:2}] {col[:75]}')
df.head(3)


## 3. Mendefinisikan Kelompok Kolom

> Dari 18 pertanyaan survei, hanya **8 pertanyaan berskala Likert** yang masuk ke analisis korelasi.
> Sisanya (kategorikal, multiple choice) digunakan untuk analisis deskriptif dan visualisasi profil.


In [ ]:
# ======================================================
# Supaya analisis lebih mudah, aku kelompokkan kolom-kolom
# menjadi beberapa grup. Nama kolomnya panjang banget dari
# Google Form, jadi aku bikin label pendek buat grafik.
# ======================================================

# --- Kolom identitas & pertanyaan kategorikal ---
COL_GENDER    = 'Jenis kelamin'
COL_SEMESTER  = 'Semester '
COL_PRODI     = 'Program studi (prodi)'
COL_FREKTUGAS = 'Seberapa sering kamu mendapatkan tugas dalam satu minggu?  '
COL_DURASI    = ' Dalam sehari, berapa rata-rata waktu yang kamu habiskan untuk mengerjakan tugas kuliah?  '
COL_FAKTOR    = 'Apa yang paling membuat beban akademik terasa berat bagi kamu? (Boleh pilih lebih dari satu)  '
COL_STRES     = 'Seberapa sering kamu merasa stres atau cemas karena urusan kuliah?  '

# --- Kolom Beban Akademik (skala Likert 1-5) ---
beban_cols = [
    'Saya merasa jumlah tugas kuliah yang diberikan terlalu banyak. ',
    'Saya sering merasa kewalahan dengan deadline tugas yang berdekatan. ',
    'Materi perkuliahan yang sulit dipahami menambah beban pikiran saya saat mengerjakan tugas. '
]
beban_labels = ['Tugas terlalu banyak', 'Kewalahan deadline', 'Materi sulit']

# --- Kolom Stabilitas Emosional (skala Likert 1-5) ---
emosi_cols = [
    'Saya mudah merasa mudah tersinggung saat sedang banyak tugas. ',
    'Suasana hati (mood) saya menjadi sangat tidak stabil ketika tugas kuliah mulai menumpuk. ',
    'Beban akademik yang tinggi membuat kualitas tidur atau waktu istirahat saya memburuk.',
    'Saya merasa tekanan akademik saat ini sangat mempengaruhi kondisi emosional saya secara keseluruhan.',
    'Tekanan akademik yang saya alami membuat saya kehilangan minat atau semangat dalam melakukan hobi/aktivitas yang biasanya saya sukai. '
]
emosi_labels = [
    'Mudah tersinggung',
    'Mood tidak stabil',
    'Kualitas tidur memburuk',
    'Pengaruh ke emosi keseluruhan',
    'Kehilangan minat hobi'
]

# --- Buat skor gabungan per responden (rata-rata Likert) ---
df['skor_beban'] = df[beban_cols].mean(axis=1)
df['skor_emosi'] = df[emosi_cols].mean(axis=1)

print('Kolom berhasil dikelompokkan!')
print(f'  {len(beban_cols)} indikator beban akademik')
print(f'  {len(emosi_cols)} indikator stabilitas emosional')


## 3a. Garis Besar Pertanyaan Survei (18 Pertanyaan)

Sebelum mendefinisikan kolom, berikut adalah peta lengkap semua pertanyaan dalam survei.
Tidak semua pertanyaan masuk ke analisis korelasi — tergantung **jenis data dan fungsinya**.

---

### 👤 Bagian A — Identitas Responden *(4 pertanyaan)*
Digunakan untuk **analisis profil responden** (pie chart, bar chart).
Tidak masuk uji korelasi karena bersifat kategorikal/nominal.

| No | Pertanyaan | Jenis Data |
|:--:|---|---|
| 1 | Nama lengkap (opsional) | Teks |
| 2 | Program studi (prodi) | Kategorikal |
| 3 | Semester | Ordinal |
| 4 | Jenis kelamin | Kategorikal |

---

### 📚 Bagian B — Beban Akademik *(5 pertanyaan)*
Dibagi jadi dua:
- **2 pertanyaan kategorikal** → dipakai untuk analisis frekuensi (bar chart)
- **3 pertanyaan Likert 1–5** → masuk ke **variabel Beban Akademik** untuk uji korelasi ✅

| No | Pertanyaan | Jenis Data | Masuk Korelasi? |
|:--:|---|---|:---:|
| 5 | Seberapa sering kamu mendapatkan tugas dalam satu minggu? | Kategorikal | ❌ |
| 6 | Berapa rata-rata waktu yang kamu habiskan untuk mengerjakan tugas per hari? | Kategorikal | ❌ |
| 7 | Saya merasa jumlah tugas kuliah yang diberikan terlalu banyak | Likert 1–5 | ✅ |
| 8 | Saya sering merasa kewalahan dengan deadline tugas yang berdekatan | Likert 1–5 | ✅ |
| 9 | Materi perkuliahan yang sulit dipahami menambah beban pikiran saya | Likert 1–5 | ✅ |

---

### ⚡ Bagian C — Faktor Beban Terberat *(1 pertanyaan)*
Pertanyaan **multiple choice** (boleh pilih lebih dari satu).
Dipakai untuk analisis frekuensi faktor, **tidak masuk korelasi** karena bukan skala numerik.

| No | Pertanyaan | Jenis Data | Masuk Korelasi? |
|:--:|---|---|:---:|
| 10 | Apa yang paling membuat beban akademik terasa berat? *(pilihan: banyak tugas, deadline mepet, materi sulit, kurang istirahat, tuntutan IPK)* | Multiple Choice | ❌ |

---

### 😟 Bagian D — Stabilitas Emosional *(8 pertanyaan)*
Dibagi jadi dua:
- **1 pertanyaan kategorikal** → dipakai untuk bar chart frekuensi stres
- **5 pertanyaan Likert 1–5** → masuk ke **variabel Stabilitas Emosional** untuk uji korelasi ✅
- **2 pertanyaan Likert tambahan** → analisis deskriptif (prioritas kesehatan mental)

| No | Pertanyaan | Jenis Data | Masuk Korelasi? |
|:--:|---|---|:---:|
| 11 | Seberapa sering kamu merasa stres atau cemas karena urusan kuliah? | Kategorikal | ❌ |
| 12 | Saya mudah merasa tersinggung saat sedang banyak tugas | Likert 1–5 | ✅ |
| 13 | Suasana hati (mood) saya menjadi tidak stabil ketika tugas menumpuk | Likert 1–5 | ✅ |
| 14 | Beban akademik yang tinggi membuat kualitas tidur saya memburuk | Likert 1–5 | ✅ |
| 15 | Tekanan akademik sangat mempengaruhi kondisi emosional saya secara keseluruhan | Likert 1–5 | ✅ |
| 16 | Tekanan akademik membuat saya kehilangan minat pada hobi/aktivitas yang saya sukai | Likert 1–5 | ✅ |
| 17 | Saya memprioritaskan kesehatan mental di tengah beban akademik | Likert 1–5 | ❌ |
| 18 | Apakah kamu merasa perlu bantuan konseling/dukungan psikologis? | Likert 1–5 | ❌ |

---

> ### 🔎 Kenapa yang masuk korelasi hanya 8 dari 18 pertanyaan?
> Uji korelasi Spearman hanya bisa bekerja dengan data **numerik ordinal** (skala Likert).
> Pertanyaan kategorikal, nominal, dan multiple choice tidak bisa langsung diuji korelasinya.
> Maka dari 18 pertanyaan, hanya **3 indikator beban + 5 indikator emosi = 8 pertanyaan Likert**
> yang dirangkum menjadi 2 skor variabel (skor_beban & skor_emosi) untuk diuji korelasinya.


In [ ]:
# ======================================================
# Supaya lebih jelas, aku cetak ringkasan pemetaan
# semua 18 pertanyaan ke dalam kelompoknya masing-masing.
# Ini juga jadi jawaban kenapa yang dipakai di korelasi
# cuma 8 pertanyaan dari total 18 pertanyaan survei.
# ======================================================

print('=' * 65)
print('PETA 18 PERTANYAAN SURVEI')
print('=' * 65)

bagian = {
    'A — Identitas Responden (4 pertanyaan)': {
        'Nama lengkap (opsional)'           : 'Teks       → Tidak dianalisis',
        'Program studi'                     : 'Kategorik  → Profil Responden',
        'Semester'                          : 'Kategorik  → Profil Responden',
        'Jenis kelamin'                     : 'Kategorik  → Profil Responden',
    },
    'B — Beban Akademik (5 pertanyaan)': {
        'Frekuensi tugas per minggu'         : 'Kategorik  → Bar chart frekuensi',
        'Durasi mengerjakan tugas per hari'  : 'Kategorik  → Bar chart frekuensi',
        'Tugas terlalu banyak [Likert]'      : 'Likert 1-5 → ✅ MASUK KORELASI (beban_cols)',
        'Kewalahan deadline [Likert]'        : 'Likert 1-5 → ✅ MASUK KORELASI (beban_cols)',
        'Materi sulit [Likert]'              : 'Likert 1-5 → ✅ MASUK KORELASI (beban_cols)',
    },
    'C — Faktor Beban Terberat (1 pertanyaan)': {
        'Faktor terberat (multiple choice)'  : 'Multiple   → Bar chart faktor',
    },
    'D — Stabilitas Emosional (8 pertanyaan)': {
        'Frekuensi stres/cemas'              : 'Kategorik  → Bar chart frekuensi stres',
        'Mudah tersinggung [Likert]'         : 'Likert 1-5 → ✅ MASUK KORELASI (emosi_cols)',
        'Mood tidak stabil [Likert]'         : 'Likert 1-5 → ✅ MASUK KORELASI (emosi_cols)',
        'Kualitas tidur memburuk [Likert]'   : 'Likert 1-5 → ✅ MASUK KORELASI (emosi_cols)',
        'Pengaruh ke emosi keseluruhan [L]'  : 'Likert 1-5 → ✅ MASUK KORELASI (emosi_cols)',
        'Kehilangan minat hobi [Likert]'     : 'Likert 1-5 → ✅ MASUK KORELASI (emosi_cols)',
        'Prioritas kesehatan mental [L]'     : 'Likert 1-5 → Analisis deskriptif',
        'Perlu bantuan konseling [Likert]'   : 'Likert 1-5 → Analisis deskriptif',
    },
}

total = 0
masuk_korelasi = 0
for nama_bagian, pertanyaan_dict in bagian.items():
    print(f'\n  BAGIAN {nama_bagian}')
    print('  ' + '-' * 61)
    for pertanyaan, keterangan in pertanyaan_dict.items():
        marker = '  ✅' if 'MASUK KORELASI' in keterangan else '   '
        print(f'{marker}  {pertanyaan:<38} | {keterangan}')
        total += 1
        if 'MASUK KORELASI' in keterangan:
            masuk_korelasi += 1

print()
print('=' * 65)
print(f'  Total pertanyaan survei       : {total} pertanyaan')
print(f'  Masuk analisis korelasi       : {masuk_korelasi} pertanyaan (Likert)')
print(f'  Tidak masuk korelasi          : {total - masuk_korelasi} pertanyaan (kategorik/lain)')
print('=' * 65)
print()
print('  CATATAN:')
print('  Uji korelasi Spearman butuh data NUMERIK ORDINAL (Likert).')
print('  8 pertanyaan Likert dirata-rata menjadi 2 skor variabel:')
print(f'    → skor_beban : rata-rata dari {len(beban_cols)} indikator beban (B7–B9)')
print(f'    → skor_emosi : rata-rata dari {len(emosi_cols)} indikator emosi (D12–D16)')
print('  Kedua skor inilah yang diuji korelasinya.')
print('=' * 65)


## 4. Profil Responden

> **Populasi:** 2.039 mahasiswa aktif ITSB  
> **Sampel:** 50 responden (Convenience Sampling)  
> **MoE:** ±14,1% — hasil bersifat eksplorasi, belum bisa digeneralisasi ke seluruh ITSB.


In [ ]:
# ======================================================
# Aku tampilin dulu konteks sampling penelitian ini.
# Penting banget untuk jujur soal keterbatasan data,
# karena ini nunjukin aku paham metodologi penelitian.
# ======================================================

POPULASI  = 2039   # Total mahasiswa ITSB
SAMPEL    = len(df)  # Yang ngisi survei
pct_sampel = round(SAMPEL / POPULASI * 100, 2)

# Margin of Error pakai rumus Slovin (buat info)
# e = sqrt(N / (sampel * (N-1) + N)) -- pendekatan
import math
margin_error = round(1 / math.sqrt(SAMPEL) * 100, 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

# Tabel info sampling
tabel_data = [
    ['Populasi (total mahasiswa ITSB)', f'{POPULASI:,} orang'],
    ['Sampel (responden survei)',        f'{SAMPEL} orang'],
    ['Persentase sampel',               f'{pct_sampel}% dari populasi'],
    ['Estimasi Margin of Error',        f'+/- {margin_error}% (confidence 95%)'],
    ['Teknik Sampling',                 'Convenience Sampling (non-probability)'],
    ['Keterbatasan',                    'Hasil tidak bisa digeneralisasi ke seluruh ITSB'],
]

tabel = ax.table(
    cellText=tabel_data,
    colLabels=['Keterangan', 'Nilai'],
    cellLoc='left', loc='center',
    colWidths=[0.5, 0.5]
)
tabel.auto_set_font_size(False)
tabel.set_fontsize(11)
tabel.scale(1, 2.2)

# Warnai header
for j in range(2):
    tabel[0, j].set_facecolor('#6c5ce7')
    tabel[0, j].set_text_props(color='white', fontweight='bold')
# Warnai baris alternating
for i in range(1, len(tabel_data)+1):
    for j in range(2):
        if i % 2 == 0:
            tabel[i, j].set_facecolor('#f0f0f0')
# Warnai baris 'Keterbatasan' merah muda
for j in range(2):
    tabel[len(tabel_data), j].set_facecolor('#ffe0e0')

ax.set_title('Konteks Sampling Penelitian', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print('=' * 58)
print('CATATAN PENTING - SAMPLING')
print('=' * 58)
print(f'  Sampel {SAMPEL} dari {POPULASI} mahasiswa = {pct_sampel}% populasi')
print(f'  Margin of Error estimasi : +/- {margin_error}%')
print()
print('  Ini cukup untuk analisis eksplorasi & deskriptif,')
print('  tapi untuk generalisasi ke seluruh ITSB diperlukan')
print('  minimal ~335 responden (rumus Slovin, e=5%).')


---
> ### 💡 Interpretasi — Konteks Sampling
>
> Dari **2.039 mahasiswa aktif ITSB**, penelitian ini mengumpulkan **50 responden** (sekitar **2,45% dari total populasi**).
>
> - **Margin of Error ±14,1%** → hasil survei bisa meleset hingga 14 poin dari kondisi nyata seluruh mahasiswa ITSB.
> - **Convenience Sampling** berarti responden dipilih berdasarkan kemudahan akses (bukan acak), sehingga hasilnya **tidak bisa digeneralisasi** ke seluruh kampus.
> - Meskipun begitu, data ini tetap valid untuk **analisis eksplorasi** — menangkap pola awal dan kecenderungan umum yang layak diteliti lebih lanjut dengan sampel lebih besar.
>
> 📌 **Intinya:** Hasil penelitian ini adalah gambaran awal yang informatif, bukan kesimpulan final untuk seluruh ITSB.



In [ ]:
# ======================================================
# Pertama aku lihat dulu siapa aja yang ngisi survei ini:
# jenis kelamin, semester, dan program studi mereka.
# ======================================================

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Profil 50 Responden Survei', fontsize=16, fontweight='bold', y=1.02)

# Jenis Kelamin - Pie chart
gender = df[COL_GENDER].value_counts()
wedges, texts, autotexts = axes[0].pie(
    gender, labels=gender.index, autopct='%1.1f%%',
    colors=['#f8a5c2','#74b9ff'], startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight('bold')
axes[0].set_title('Jenis Kelamin')

# Semester - Bar chart
semester = df[COL_SEMESTER].value_counts().sort_index()
bars = axes[1].bar(semester.index, semester.values,
                   color=['#6c5ce7','#a29bfe','#dfe6e9'],
                   edgecolor='white', linewidth=1.5)
axes[1].set_title('Distribusi Semester')
axes[1].set_xlabel('Semester')
axes[1].set_ylabel('Jumlah Responden')
axes[1].set_ylim(0, max(semester.values) * 1.25)
for bar in bars:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., h + 0.4,
                 f'{int(h)} orang', ha='center', fontweight='bold', fontsize=10)

# Program Studi - Horizontal bar
import numpy as np
prodi = df[COL_PRODI].value_counts()
colors_p = plt.cm.Set3(np.linspace(0, 1, len(prodi)))
axes[2].barh(prodi.index, prodi.values, color=colors_p, edgecolor='white', linewidth=1.5)
axes[2].set_title('Program Studi')
axes[2].set_xlabel('Jumlah Responden')
axes[2].set_xlim(0, max(prodi.values) * 1.3)
for i, (val, name) in enumerate(zip(prodi.values, prodi.index)):
    axes[2].text(val + 0.1, i, f'{val} orang', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Insight otomatis
gender_mayoritas = gender.idxmax()
pct_gender = round(gender.max() / gender.sum() * 100, 1)
print('=' * 55)
print('INSIGHT - Profil Responden')
print('=' * 55)
print(f'  Mayoritas responden   : {gender_mayoritas} ({pct_gender}%)')
print(f'  Semester 1-2          : {df[COL_SEMESTER].value_counts().iloc[0]} dari 50 orang')
print(f'  Prodi terbanyak       : {prodi.idxmax()} ({prodi.max()} orang)')
print()
print('  --> Kebanyakan responden adalah mahasiswa baru (sem 1-2),')
print('      yang masih dalam tahap adaptasi dunia perkuliahan.')


---
> ### 💡 Interpretasi — Profil Responden
>
> Visualisasi ini memperlihatkan siapa saja yang mengisi survei:
>
> - **Jenis Kelamin:** Mayoritas responden adalah **perempuan** — umum terjadi karena partisipasi perempuan dalam survei akademik umumnya lebih tinggi.
> - **Semester:** Sebagian besar berasal dari **semester awal (1–2)**, artinya mereka masih dalam fase adaptasi perkuliahan — kelompok yang paling rentan merasakan tekanan akademik baru.
> - **Program Studi:** Responden didominasi satu atau dua prodi, sehingga hasil lebih mencerminkan kondisi prodi tersebut daripada seluruh ITSB secara merata.
>
> 📌 **Intinya:** Kita terutama meneliti mahasiswa baru perempuan dari prodi mayoritas — penting diingat saat menafsirkan hasil akhirnya.



## 5. Analisis Beban Akademik


In [ ]:
# ======================================================
# Di sini aku lihat seberapa berat beban akademik yang
# dirasakan responden dari 3 indikator, skala 1-5.
# Skor >= 3.5 aku kategorikan sebagai beban TINGGI.
# ======================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Analisis Beban Akademik Mahasiswa', fontsize=15, fontweight='bold')

# Bar chart: Rata-rata per Indikator
rata_beban = df[beban_cols].mean().round(2)
warna_bar  = ['#e17055', '#fdcb6e', '#0984e3']
bars = axes[0].bar(beban_labels, rata_beban.values,
                   color=warna_bar, edgecolor='white', linewidth=2, width=0.5)
axes[0].set_ylim(0, 5.8)
axes[0].set_ylabel('Rata-rata Skor (1-5)')
axes[0].set_title('Rata-rata Skor per Indikator Beban Akademik')
axes[0].axhline(y=3, color='gray', linestyle='--', alpha=0.6, label='Batas tengah (3.0)')
axes[0].legend(fontsize=9)
kat_map = {1:'Sangat Rendah', 2:'Rendah', 3:'Sedang', 4:'Tinggi', 5:'Sangat Tinggi'}
for bar, val in zip(bars, rata_beban.values):
    kat = kat_map.get(round(val), 'Sedang')
    axes[0].text(bar.get_x() + bar.get_width()/2., val + 0.1,
                 f'{val}\n({kat})', ha='center', fontsize=10, fontweight='bold')

# Bar chart: Frekuensi tugas per minggu
freq_order = ['Tidak pernah', '1-2 kali', '3-5 kali', 'Lebih dari 5 kali']
freq_data  = df[COL_FREKTUGAS].value_counts()
# Normalisasi tanda strip agar cocok
freq_data.index = freq_data.index.str.replace('\u20131', '-1').str.replace('\u20135', '-5')
freq_data = freq_data.reindex(freq_order).dropna()
warna_freq = ['#55efc4', '#fdcb6e', '#e17055', '#d63031']
bars2 = axes[1].bar(freq_data.index, freq_data.values,
                    color=warna_freq[:len(freq_data)],
                    edgecolor='white', linewidth=2, width=0.5)
axes[1].set_title('Frekuensi Mendapat Tugas per Minggu')
axes[1].set_ylabel('Jumlah Responden')
axes[1].set_ylim(0, max(freq_data.values) * 1.3)
for bar in bars2:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., h + 0.3,
                 f'{int(h)} orang', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

# Insight otomatis
top_idx = list(rata_beban.values).index(max(rata_beban.values))
skor_total_beban = df['skor_beban'].mean().round(2)
kat_total = 'TINGGI' if skor_total_beban >= 3.5 else ('SEDANG' if skor_total_beban >= 2.5 else 'RENDAH')

print('=' * 55)
print('INSIGHT - Beban Akademik')
print('=' * 55)
print(f'  Indikator tertinggi              : {beban_labels[top_idx]} ({max(rata_beban.values)}/5)')
print(f'  Rata-rata beban keseluruhan      : {skor_total_beban}/5')
print(f'  Kategori                         : {kat_total}')
print()
print('  --> Mahasiswa merasakan beban akademik yang cukup berat,')
print('      terutama dari materi yang sulit dan deadline mepet.')


---
> ### 💡 Interpretasi — Analisis Beban Akademik
>
> **Grafik kiri** menampilkan rata-rata skor tiap indikator beban (skala 1–5):
> - Skor **> 3,0** → mahasiswa cenderung *setuju* bahwa ini menjadi beban.
> - Indikator dengan skor tertinggi = yang paling banyak dikeluhkan.
>
> **Grafik kanan** menunjukkan frekuensi tugas per minggu:
> - Semakin banyak responden di "3–5 kali" atau "lebih dari 5 kali" → beban mingguan semakin tinggi.
>
> 📌 **Intinya:** Mahasiswa ITSB merasakan beban akademik yang **cukup berat**, terutama akibat materi yang sulit dipahami dan deadline yang saling bertumpukan.



## 6. Faktor Penyebab Beban Akademik


In [ ]:
# ======================================================
# Pertanyaan ini boleh pilih lebih dari satu (multiple
# choice), jadi aku pisahkan dengan split(',') lalu hitung
# frekuensi masing-masing faktor yang disebutkan.
# ======================================================

all_faktor = []
for row in df[COL_FAKTOR].dropna():
    items = [x.strip() for x in row.split(',')]
    all_faktor.extend(items)

faktor_count = pd.Series(all_faktor).value_counts()
# Rapikan nama yang terpotong dari Google Form
faktor_count.index = faktor_count.index.str.replace(
    'Tuntutan nilai/IPK tingg', 'Tuntutan nilai/IPK tinggi'
)

plt.figure(figsize=(12, 5))
colors_f = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(faktor_count)))
bars = plt.barh(faktor_count.index[::-1], faktor_count.values[::-1],
               color=colors_f, edgecolor='white', linewidth=1.5)
plt.xlabel('Jumlah Responden yang Memilih')
plt.title('Faktor yang Paling Membuat Beban Akademik Berat\n(Responden boleh memilih lebih dari satu)')
for bar, val in zip(bars, faktor_count.values[::-1]):
    plt.text(val + 0.2, bar.get_y() + bar.get_height()/2.,
             f'{val} orang', va='center', fontsize=10, fontweight='bold')
plt.xlim(0, max(faktor_count.values) * 1.25)
plt.tight_layout()
plt.show()

print('=' * 55)
print('INSIGHT - Faktor Penyebab Beban Akademik')
print('=' * 55)
for i, (faktor, jumlah) in enumerate(faktor_count.head(3).items(), 1):
    pct = round(jumlah / len(df) * 100, 1)
    print(f'  {i}. {faktor} --> {jumlah} orang ({pct}%)')
print()
print('  --> Ketiga faktor ini mendominasi keluhan akademik mahasiswa.')
print('      Ini bisa jadi masukan penting buat strategi belajar.')


---
> ### 💡 Interpretasi — Faktor Penyebab Beban Akademik
>
> Grafik ini menunjukkan **apa yang paling sering dianggap memberatkan** oleh mahasiswa (boleh pilih lebih dari satu, sehingga total pilihan bisa melebihi 50 orang):
>
> - Faktor dengan bar terpanjang = masalah yang paling banyak dirasakan.
> - Satu mahasiswa bisa merasakan beberapa faktor sekaligus.
>
> 📌 **Intinya:** Tekanan akademik bersifat **berlapis** — bukan dari satu sumber saja, melainkan kombinasi dari banyak tugas, deadline mepet, dan materi sulit yang datang bersamaan.



## 7. Analisis Stabilitas Emosional


In [ ]:
# ======================================================
# Sekarang aku lihat sisi emosional - ada 5 indikator
# yang mengukur seberapa parah dampak beban akademik
# ke kondisi psikologis mahasiswa (skala Likert 1-5).
# ======================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Analisis Stabilitas Emosional Mahasiswa', fontsize=15, fontweight='bold')

# Horizontal bar: Rata-rata per Indikator
rata_emosi = df[emosi_cols].mean().round(2)
warna_emosi = ['#fd79a8', '#e84393', '#6c5ce7', '#a29bfe', '#fdcb6e']
bars = axes[0].barh(emosi_labels, rata_emosi.values,
                    color=warna_emosi, edgecolor='white', linewidth=1.5)
axes[0].set_xlabel('Rata-rata Skor (1-5)')
axes[0].set_title('Skor Dampak Beban Akademik ke Emosi')
axes[0].axvline(x=3, color='gray', linestyle='--', alpha=0.6, label='Batas tengah (3.0)')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 6)
for bar, val in zip(bars, rata_emosi.values):
    axes[0].text(val + 0.1, bar.get_y() + bar.get_height()/2.,
                 f'{val}', va='center', fontsize=11, fontweight='bold')

# Bar: Frekuensi stres
stres_order = ['Tidak pernah', 'Jarang', 'Kadang-kadang', 'Sering', 'Sangat sering']
stres_data = df[COL_STRES].value_counts().reindex(stres_order).fillna(0)
warna_stres = ['#55efc4', '#74b9ff', '#fdcb6e', '#e17055', '#d63031']
bars2 = axes[1].bar(stres_data.index, stres_data.values,
                    color=warna_stres, edgecolor='white', linewidth=2, width=0.55)
axes[1].set_title('Seberapa Sering Mahasiswa Merasa Stres?')
axes[1].set_ylabel('Jumlah Responden')
axes[1].set_ylim(0, max(stres_data.values) * 1.3)
for bar in bars2:
    h = bar.get_height()
    if h > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2., h + 0.2,
                     f'{int(h)} orang', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

top_emosi_idx = list(rata_emosi.values).index(max(rata_emosi.values))
skor_emosi_total = df['skor_emosi'].mean().round(2)
pct_stres = round((stres_data.get('Sering', 0) + stres_data.get('Sangat sering', 0)) / len(df) * 100, 1)

print('=' * 55)
print('INSIGHT - Stabilitas Emosional')
print('=' * 55)
print(f'  Indikator dampak tertinggi   : {emosi_labels[top_emosi_idx]} ({max(rata_emosi.values)}/5)')
print(f'  Rata-rata skor emosional     : {skor_emosi_total}/5')
print(f'  Sering/Sangat sering stres   : {pct_stres}% dari seluruh responden')
print()
print('  --> Gangguan kualitas tidur menjadi dampak emosional')
print('      terbesar dari tekanan akademik yang dirasakan.')


---
> ### 💡 Interpretasi — Analisis Stabilitas Emosional
>
> **Grafik kiri** menampilkan rata-rata skor dampak emosional dari 5 indikator (skala 1–5):
> - Skor tinggi menandakan dampak emosional yang signifikan dari beban akademik.
> - Indikator **kualitas tidur memburuk** dan **mood tidak stabil** mendapat skor tertinggi.
>
> **Grafik kanan** menunjukkan frekuensi stres/kecemasan yang dilaporkan sendiri:
> - Semakin banyak yang di kategori "Sering" dan "Sangat sering" → tekanan dirasakan hampir setiap hari.
>
> 📌 **Intinya:** Beban akademik tidak sekadar melelahkan secara fisik — ia berdampak nyata pada **kesehatan mental**, terutama lewat gangguan tidur dan ketidakstabilan suasana hati.



## 8. Distribusi Data: Histogram & Boxplot

> Sebelum uji korelasi, kita perlu lihat **bentuk sebaran data** tiap indikator.  
> - **Histogram** = lihat apakah data condong ke kiri/kanan atau simetris  
> - **Boxplot** = lihat nilai tengah, rentang, dan apakah ada nilai ekstrem (outlier)  

Ini penting karena **metode korelasi yang tepat** tergantung dari normalitas data.


In [ ]:
# ======================================================
# Visualisasi distribusi jawaban tiap indikator.
# Aku pakai histogram + kurva normal sebagai pembanding,
# plus boxplot buat lihat sebaran dan outlier-nya.
# ======================================================

beban_labels_short = ['Tugas\nterlalu banyak', 'Kewalahan\ndeadline', 'Materi\nsulit']
emosi_labels_short = ['Mudah\ntersinggung', 'Mood\ntdk stabil', 'Tidur\nmemburuk',
                      'Pengaruh\nke emosi', 'Minat\nhobi hilang']

# === HISTOGRAM per indikator ===
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Distribusi Jawaban per Indikator (Skala Likert 1-5)', fontsize=15, fontweight='bold')

warna_beban = ['#e17055', '#fdcb6e', '#0984e3']
warna_emosi = ['#fd79a8', '#e84393', '#6c5ce7', '#a29bfe', '#fdcb6e']

# Baris 1: Beban Akademik
for i, (col, label, warna) in enumerate(zip(beban_cols, beban_labels_short, warna_beban)):
    ax = axes[0][i]
    data = df[col]
    counts = data.value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=warna, edgecolor='white',
           linewidth=1.5, alpha=0.85, width=0.6)
    ax.set_title(f'[BEBAN]\n{label}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Skor (1-5)')
    ax.set_ylabel('Frekuensi')
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_xlim(0.5, 5.5)
    # Tambah nilai mean & std
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean={data.mean():.2f}')
    ax.legend(fontsize=8)
    # Label angka di atas bar
    for skor, cnt in counts.items():
        ax.text(skor, cnt + 0.1, str(cnt), ha='center', fontsize=9, fontweight='bold')

# Baris 2: Stabilitas Emosional
for i, (col, label, warna) in enumerate(zip(emosi_cols, emosi_labels_short, warna_emosi)):
    ax = axes[1][i]
    data = df[col]
    counts = data.value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=warna, edgecolor='white',
           linewidth=1.5, alpha=0.85, width=0.6)
    ax.set_title(f'[EMOSI]\n{label}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Skor (1-5)')
    ax.set_ylabel('Frekuensi')
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_xlim(0.5, 5.5)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean={data.mean():.2f}')
    ax.legend(fontsize=8)
    for skor, cnt in counts.items():
        ax.text(skor, cnt + 0.1, str(cnt), ha='center', fontsize=9, fontweight='bold')

# Sembunyikan axes kosong (beban cuma 3 kolom)
for j in range(3, 5):
    axes[0][j].axis('off')

plt.tight_layout()
plt.show()

# === BOXPLOT gabungan ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Boxplot: Sebaran & Outlier Tiap Indikator', fontsize=14, fontweight='bold')

# Boxplot Beban
data_beban = [df[col].values for col in beban_cols]
bp1 = axes[0].boxplot(data_beban, labels=beban_labels_short,
                       patch_artist=True, notch=False,
                       medianprops={'color': 'red', 'linewidth': 2})
for patch, color in zip(bp1['boxes'], warna_beban):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title('Indikator Beban Akademik')
axes[0].set_ylabel('Skor (1-5)')
axes[0].set_ylim(0, 6)
axes[0].axhline(y=3, color='gray', linestyle=':', alpha=0.6)

# Boxplot Emosi
data_emosi = [df[col].values for col in emosi_cols]
bp2 = axes[1].boxplot(data_emosi, labels=emosi_labels_short,
                       patch_artist=True, notch=False,
                       medianprops={'color': 'red', 'linewidth': 2})
for patch, color in zip(bp2['boxes'], warna_emosi):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Indikator Stabilitas Emosional')
axes[1].set_ylabel('Skor (1-5)')
axes[1].set_ylim(0, 6)
axes[1].axhline(y=3, color='gray', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

print('=' * 58)
print('CARA BACA BOXPLOT:')
print('=' * 58)
print('  Garis merah tengah = nilai MEDIAN (nilai tengah data)')
print('  Kotak biru         = 50% data berada di rentang ini')
print('  Garis panjang      = rentang data (min-max normal)')
print('  Titik di luar      = OUTLIER (nilai yang sangat berbeda)')
print()
print('  Semakin simetris kotaknya = semakin normal distribusinya.')
print('  Kotak yang miring ke atas/bawah = data condong (skewed).')


---
> ### 💡 Interpretasi — Distribusi Data (Histogram & Boxplot)
>
> **Histogram** menunjukkan berapa banyak mahasiswa yang menjawab tiap nilai (1–5):
> - Puncak di kanan (nilai 4–5) → mayoritas setuju / merasakan dampak tersebut.
> - Garis merah putus-putus = nilai rata-rata (mean) tiap indikator.
>
> **Boxplot** merangkum distribusi data dalam satu kotak:
> - **Garis merah** di tengah = nilai tengah (median).
> - **Kotak** = rentang di mana 50% responden berada.
> - **Garis panjang** ke atas/bawah = rentang normal (min–max).
> - **Titik di luar garis** = outlier (jawaban yang sangat berbeda dari mayoritas).
>
> 📌 **Intinya:** Mayoritas mahasiswa menjawab di skor **3–5**, menandakan beban akademik dan dampak emosionalnya **dirasakan cukup hingga sangat tinggi**. Distribusi yang tidak sempurna simetris ini menjadi alasan digunakannya uji non-parametrik (Spearman) di tahap berikutnya.



## 9. Uji Normalitas: Syarat Sebelum Korelasi

---

### 📌 Apa itu Uji Normalitas?
Uji normalitas adalah langkah **wajib** sebelum memilih metode korelasi.
Tujuannya untuk mengetahui apakah data kita berdistribusi normal atau tidak,
karena pemilihan metode korelasi bergantung pada hasil uji ini.

---

### 🔬 Uji Shapiro-Wilk
Kami menggunakan **Uji Shapiro-Wilk** karena:
- Paling akurat untuk ukuran sampel **kecil hingga sedang (n < 100)**
- Lebih *powerful* dibanding uji Kolmogorov-Smirnov untuk n kecil
- Sampel kami n = 50, sehingga Shapiro-Wilk adalah pilihan yang paling tepat

**Rumus Statistik Shapiro-Wilk:**

$$W = \frac{\left(\sum_{i=1}^{n} a_i x_{(i)}\right)^2}{\sum_{i=1}^{n} (x_i - \bar{x})^2}$$

Di mana:
- $x_{(i)}$ = nilai data yang sudah diurutkan (order statistics)
- $a_i$ = koefisien yang diturunkan dari rata-rata, variansi, dan kovarians order statistics distribusi normal
- Nilai W mendekati **1** → data mendekati distribusi normal
- Nilai W jauh dari 1 → data menyimpang dari normal

---

### 📐 Hipotesis Uji Shapiro-Wilk

| Hipotesis | Pernyataan |
|---|---|
| **H₀ (Hipotesis Nol)** | Data berasal dari populasi yang berdistribusi **normal** |
| **H₁ (Hipotesis Alternatif)** | Data **tidak** berasal dari populasi yang berdistribusi normal |

---

### ⚖️ Kriteria Pengambilan Keputusan (α = 0,05)

| Kondisi | Keputusan | Interpretasi |
|---|---|---|
| p-value **≥ 0,05** | **Gagal Tolak H₀** | Data berdistribusi normal → gunakan **Pearson** |
| p-value **< 0,05** | **Tolak H₀** | Data tidak normal → gunakan **Spearman** |

---

### 📊 Cara Membaca QQ-Plot
QQ-Plot (*Quantile-Quantile Plot*) adalah visualisasi pendukung uji normalitas:
- **Titik-titik mengikuti garis diagonal merah** → data mendekati distribusi normal ✅
- **Titik-titik menyimpang jauh dari garis** → data tidak normal ❌
- Penyimpangan di ujung kiri/kanan menunjukkan data memiliki ekor (*skewed*)

> **Uji yang digunakan:** Shapiro-Wilk dengan tingkat signifikansi **α = 0,05**


In [ ]:
# ======================================================
# Uji Shapiro-Wilk untuk masing-masing variabel:
# 1. Skor Beban Akademik (rata-rata 3 indikator Likert)
# 2. Skor Stabilitas Emosional (rata-rata 5 indikator Likert)
# Hasilnya menentukan metode korelasi yang akan dipakai.
# ======================================================

from scipy import stats

# ── Langkah 1: Hitung statistik Shapiro-Wilk ─────────────────────────────
stat_beban, p_beban = stats.shapiro(df['skor_beban'])
stat_emosi, p_emosi = stats.shapiro(df['skor_emosi'])

normal_beban    = p_beban > 0.05
normal_emosi    = p_emosi > 0.05
keduanya_normal = normal_beban and normal_emosi

# ── Langkah 2: QQ-Plot untuk visualisasi normalitas ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    'Q-Q Plot: Visualisasi Normalitas Data\n'
    '(Titik mengikuti garis diagonal = data normal)',
    fontsize=13, fontweight='bold'
)

for ax, data, label, p_val, w_val, is_norm in zip(
    axes,
    [df['skor_beban'], df['skor_emosi']],
    ['Skor Beban Akademik', 'Skor Stabilitas Emosional'],
    [p_beban, p_emosi],
    [stat_beban, stat_emosi],
    [normal_beban, normal_emosi]
):
    (osm, osr), (slope, intercept, r_qq) = stats.probplot(data, dist='norm')

    # Plot titik data
    ax.scatter(osm, osr, color='#6c5ce7', alpha=0.75,
               s=60, edgecolor='white', linewidth=0.8, label='Data observasi', zorder=3)
    # Garis normal ideal
    ax.plot(osm, slope * np.array(osm) + intercept,
            'r-', linewidth=2.5, label='Garis normal ideal', zorder=2)
    # Area confidence band (95%)
    ax.fill_between(osm,
                    slope * np.array(osm) + intercept - 0.3,
                    slope * np.array(osm) + intercept + 0.3,
                    alpha=0.1, color='red', label='Band ±0.3')

    status      = 'NORMAL ✅' if is_norm else 'TIDAK NORMAL ❌'
    warna_title = '#00b894' if is_norm else '#e17055'
    keputusan   = 'Gagal Tolak H₀' if is_norm else 'Tolak H₀'

    ax.set_title(f'{label}\nW = {w_val:.4f}  |  p = {p_val:.4f}  →  {status}',
                 color=warna_title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Kuantil Teoritis (Distribusi Normal)', fontsize=10)
    ax.set_ylabel('Kuantil Data Aktual', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Kotak ringkasan hasil di pojok kiri atas
    box_color = '#d4f7dc' if is_norm else '#ffeaa7'
    ax.text(
        0.05, 0.95,
        f'W  = {w_val:.4f}\np  = {p_val:.4f}\nα  = 0,05\n{keputusan}\n→ {status}',
        transform=ax.transAxes, fontsize=10, fontweight='bold',
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor=box_color, alpha=0.92,
                  edgecolor='#636e72')
    )

plt.tight_layout()
plt.show()

# ── Langkah 3: Tabel hasil lengkap ───────────────────────────────────────
print('=' * 65)
print('HASIL UJI NORMALITAS SHAPIRO-WILK')
print('=' * 65)
print(f'  Jumlah sampel (n)    : {len(df)} responden')
print(f'  Tingkat signifikansi : α = 0,05')
print()
print(f'  {"Variabel":<30} {"W":<10} {"p-value":<12} {"Keputusan":<20} {"Ket."}')
print('  ' + '-' * 61)

for var, w, p, is_n in [
    ('Skor Beban Akademik',    stat_beban, p_beban, normal_beban),
    ('Skor Stabilitas Emosional', stat_emosi, p_emosi, normal_emosi)
]:
    kep = 'Gagal Tolak H₀' if is_n else 'Tolak H₀'
    ket = 'Normal' if is_n else 'Tidak Normal'
    print(f'  {var:<30} {w:<10.4f} {p:<12.4f} {kep:<20} {ket}')

print()
print('  INTERPRETASI:')
if keduanya_normal:
    print('  ✅ Kedua variabel berdistribusi NORMAL (p > 0,05)')
    print('  → Asumsi Pearson terpenuhi → digunakan Korelasi PEARSON')
else:
    if not normal_beban:
        print(f'  ❌ Skor Beban Akademik TIDAK normal (p={p_beban:.4f} < 0,05 → Tolak H₀)')
    if not normal_emosi:
        print(f'  ❌ Skor Stab. Emosional TIDAK normal (p={p_emosi:.4f} < 0,05 → Tolak H₀)')
    print()
    print('  → Karena asumsi normalitas tidak terpenuhi,')
    print('    Korelasi PEARSON tidak bisa digunakan.')
    print('    Digunakan Korelasi SPEARMAN (non-parametrik):')
    print('    • Tidak mensyaratkan normalitas distribusi data')
    print('    • Bekerja berdasarkan RANKING nilai, bukan nilai aslinya')
    print('    • Lebih robust untuk data Likert & distribusi non-normal')
    print('    • Tetap valid dan dapat dipercaya secara statistik')
print('=' * 65)


---
> ### 💡 Interpretasi — Uji Normalitas Shapiro-Wilk
>
> Uji ini menentukan metode statistik yang tepat untuk korelasi:
>
> - **p-value ≥ 0,05** → data berdistribusi normal → boleh pakai Korelasi **Pearson**.
> - **p-value < 0,05** → data TIDAK normal → harus pakai Korelasi **Spearman** (non-parametrik).
>
> **QQ-Plot** adalah visualisasi pendukungnya:
> - Titik-titik **mengikuti garis merah** → data mendekati normal ✅
> - Titik-titik **menyimpang jauh** dari garis → data tidak normal ❌
>
> 📌 **Intinya:** Karena data tidak berdistribusi normal (p < 0,05), penelitian ini menggunakan **Korelasi Spearman** — metode yang tepat untuk data survei skala Likert dan tidak memerlukan asumsi normalitas.



## 10. Analisis Korelasi (Dipilih Otomatis dari Uji Normalitas)

---

### 📌 Apa itu Korelasi Spearman?
**Korelasi Rank Spearman** (ρ / rho) adalah metode korelasi **non-parametrik** yang
mengukur kekuatan dan arah hubungan monoton antara dua variabel.
Berbeda dengan Pearson yang bekerja pada nilai asli, Spearman bekerja pada **ranking** dari nilai tersebut.

**Rumus Korelasi Spearman:**

$$r_s = 1 - \frac{6 \sum d_i^2}{n(n^2-1)}$$

Di mana:
- $d_i$ = selisih ranking antara dua variabel untuk responden ke-i ($d_i = rank(X_i) - rank(Y_i)$)
- $n$ = jumlah pasangan data (jumlah responden)
- Hasil $r_s$ berkisar antara **-1 hingga +1**

---

### 📐 Hipotesis Uji Korelasi

| Hipotesis | Pernyataan |
|---|---|
| **H₀** | Tidak terdapat korelasi antara beban akademik dan stabilitas emosional (ρ = 0) |
| **H₁** | Terdapat korelasi yang signifikan antara keduanya (ρ ≠ 0) |

---

### 📊 Interpretasi Nilai r (Koefisien Korelasi)

| Nilai r | Kekuatan Korelasi | Arah |
|:---:|---|---|
| 0,80 – 1,00 | **Sangat Kuat** | Positif |
| 0,60 – 0,79 | **Kuat** | Positif |
| 0,40 – 0,59 | **Sedang** | Positif |
| 0,20 – 0,39 | **Lemah** | Positif |
| 0,00 – 0,19 | **Sangat Lemah / Tidak Ada** | — |
| Negatif | (sama seperti di atas, tapi arah terbalik) | Negatif |

---

### ⚖️ Kriteria Pengambilan Keputusan

| Kondisi | Keputusan | Artinya |
|---|---|---|
| p-value **< 0,05** | **Tolak H₀** | Ada korelasi signifikan secara statistik ✅ |
| p-value **≥ 0,05** | **Gagal Tolak H₀** | Tidak cukup bukti adanya korelasi |

---

### 🔄 Kenapa Spearman Dipilih Otomatis?
Kode di bawah akan **otomatis memilih** metode yang tepat berdasarkan hasil uji normalitas:
- Jika **kedua data normal** → Pearson (parametrik)
- Jika **salah satu/keduanya tidak normal** → Spearman (non-parametrik)

> Pendekatan ini memastikan metode yang digunakan **selalu valid secara statistik**,
> tidak peduli bagaimana distribusi data yang didapat.


In [ ]:
# ======================================================
# Uji korelasi otomatis — metode dipilih berdasarkan
# hasil uji normalitas Shapiro-Wilk sebelumnya.
# Output mencakup scatter plot + interpretasi lengkap.
# ======================================================

def interpretasi_r(r):
    ar = abs(r)
    if ar >= 0.80:   return 'Sangat Kuat'
    elif ar >= 0.60: return 'Kuat'
    elif ar >= 0.40: return 'Sedang'
    elif ar >= 0.20: return 'Lemah'
    else:            return 'Sangat Lemah / Tidak Ada'

# ── Langkah 1: Pilih metode otomatis ─────────────────────────────────────
if keduanya_normal:
    r, p_value = stats.pearsonr(df['skor_beban'], df['skor_emosi'])
    metode  = 'Pearson'
    catatan = 'dipilih karena kedua data berdistribusi normal'
else:
    r, p_value = stats.spearmanr(df['skor_beban'], df['skor_emosi'])
    metode  = 'Spearman'
    catatan = 'dipilih karena salah satu/kedua data TIDAK normal'

kekuatan = interpretasi_r(r)
arah     = 'POSITIF (+)' if r > 0 else 'NEGATIF (-)'
sig      = p_value < 0.05

# ── Langkah 2: Hitung ranking Spearman (untuk ditampilkan) ───────────────
if metode == 'Spearman':
    rank_beban = df['skor_beban'].rank()
    rank_emosi = df['skor_emosi'].rank()
    d_sq_sum   = ((rank_beban - rank_emosi) ** 2).sum()
    n          = len(df)
    rs_manual  = 1 - (6 * d_sq_sum) / (n * (n**2 - 1))

# ── Langkah 3: Scatter Plot FULL ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(
    df['skor_beban'], df['skor_emosi'],
    color='#6c5ce7', alpha=0.75, s=100,
    edgecolor='white', linewidth=1.2,
    label=f'Tiap titik = 1 responden (n={len(df)})'
)

# Garis tren regresi linear
m, b_reg = np.polyfit(df['skor_beban'], df['skor_emosi'], 1)
x_line   = np.linspace(df['skor_beban'].min(), df['skor_beban'].max(), 100)
ax.plot(x_line, m * x_line + b_reg,
        color='#e17055', linewidth=2.5,
        label=f'Garis tren (slope = {m:.3f})')

# Anotasi label aksis dan judul
ax.set_xlabel('Skor Beban Akademik (rata-rata Likert 1–5)', fontsize=12)
ax.set_ylabel('Skor Dampak Emosional (rata-rata Likert 1–5)', fontsize=12)
ax.set_title(
    f'Scatter Plot: Beban Akademik vs Stabilitas Emosional\n'
    f'Korelasi {metode}  |  r = {r:.4f}  |  p = {p_value:.4f}  |  {kekuatan}',
    fontsize=13, fontweight='bold', pad=15
)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)

# Kotak ringkasan hasil di pojok kiri atas
kotak_warna = '#d4f7dc' if sig else '#ffeaa7'
info = (
    f'Metode   : {metode}\n'
    f'r        = {r:.4f}\n'
    f'p-value  = {p_value:.4f}\n'
    f'Kekuatan : {kekuatan}\n'
    f'Arah     : {arah}\n'
    f'Sig.(α=0,05): {"Ya ✅" if sig else "Tidak ❌"}'
)
ax.text(
    0.04, 0.96, info,
    transform=ax.transAxes, fontsize=10, fontweight='bold',
    verticalalignment='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor=kotak_warna,
              alpha=0.92, edgecolor='#636e72')
)

ax.set_xlim(0.8, 5.4)
ax.set_ylim(0.8, 5.4)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_yticks([1, 2, 3, 4, 5])
plt.tight_layout()
plt.show()

# ── Langkah 4: Output hasil lengkap ──────────────────────────────────────
print('=' * 65)
print(f'HASIL UJI KORELASI {metode.upper()}')
print('=' * 65)
print(f'  Jumlah sampel (n)    : {len(df)} responden')
print(f'  Metode korelasi      : {metode} ({catatan})')
print()
print('  ── Statistik Hasil ──────────────────────────────────')
print(f'  Koefisien r          : {r:.4f}')
if metode == 'Spearman':
    print(f'  r manual (rumus Σd²) : {rs_manual:.4f}  ← verifikasi rumus')
    print(f'  Σd²                  : {d_sq_sum:.2f}')
print(f'  p-value              : {p_value:.4f}')
print(f'  Tingkat signifikansi : α = 0,05')
print()
print('  ── Keputusan Statistik ──────────────────────────────')
if sig:
    print(f'  p = {p_value:.4f} < α = 0,05  →  TOLAK H₀')
    print('  → Ada korelasi yang SIGNIFIKAN secara statistik ✅')
else:
    print(f'  p = {p_value:.4f} ≥ α = 0,05  →  GAGAL TOLAK H₀')
    print('  → Tidak cukup bukti adanya korelasi yang signifikan')
print()
print('  ── Interpretasi Korelasi ────────────────────────────')
print(f'  Arah korelasi        : {arah}')
print(f'  Kekuatan korelasi    : {kekuatan}')
print()
print(f'  Makna: Terdapat hubungan {arah.split()[0]} yang {kekuatan.upper()}')
print('  antara beban akademik dan stabilitas emosional mahasiswa.')
print()
if r > 0:
    print('  Artinya: semakin TINGGI beban akademik yang dirasakan,')
    print('  semakin BESAR pula gangguan emosional yang dialami.')
else:
    print('  Artinya: semakin TINGGI beban akademik yang dirasakan,')
    print('  semakin RENDAH stabilitas emosional mahasiswa.')
if sig:
    print()
    print('  p < 0,05 → hubungan ini BUKAN kebetulan.')
    print('  Pola ini nyata dan dapat dipercaya dari data 50 responden.')
print('=' * 65)


---
> ### 💡 Interpretasi — Uji Korelasi Spearman
>
> **Scatter plot** memperlihatkan hubungan antara skor beban akademik (sumbu X) dan skor dampak emosional (sumbu Y) dari 50 mahasiswa:
> - Setiap **titik ungu** = 1 responden.
> - **Garis merah** = tren umum hubungan keduanya.
> - Garis mengarah **kanan atas** → semakin tinggi beban, semakin tinggi gangguan emosional.
>
> Cara membaca hasil korelasi:
> - **Nilai r** = kekuatan hubungan (0 = tidak ada, 1 = sempurna positif).
> - **p-value < 0,05** = hubungan ini **bukan kebetulan**, ada pola nyata di data.
>
> 📌 **Intinya:** Ada hubungan **positif yang signifikan** — **semakin berat beban akademik yang dirasakan, semakin besar pula gangguan emosional yang dialami mahasiswa ITSB**. Temuan ini valid secara statistik dan bukan sekadar kebetulan.



### 📊 Heatmap Korelasi Antar Semua Indikator

> **Cara baca heatmap ini:**
> - Setiap kotak = korelasi antara 2 indikator
> - **Biru tua** = korelasi positif kuat (semakin tinggi satu, semakin tinggi yang lain)
> - **Putih** = tidak ada korelasi
> - **Merah** = korelasi negatif
> - Hanya separuh bawah diagonal yang ditampilkan *(lower triangle)* karena nilai atas dan bawah sama


In [ ]:
# ======================================================
# Heatmap korelasi yang rapi dan mudah dibaca.
# - Lower triangle: 8 baris dan 8 kolom lengkap
# - Warna biru: putih/muda = lemah, biru tua = kuat
# - Label sumbu X sejajar dengan kotaknya
# - Legend interpretasi di bawah biar tidak bentrok
# ======================================================

# Hitung matriks korelasi Spearman
corr_df = df[beban_cols + emosi_cols].copy()
corr_df.columns = beban_labels + emosi_labels
corr_matrix = corr_df.corr(method='spearman' if not keduanya_normal else 'pearson')

# Mask HANYA upper triangle (k=1) — diagonal TETAP tampil
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(12, 9))
fig.patch.set_facecolor('#FAFAFA')

# Warna Blues: putih = lemah, biru tua = kuat
# vmin=0, vmax=1 karena semua korelasi kita positif
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    vmin=0, vmax=1,
    ax=ax,
    linewidths=1.2,
    linecolor='white',
    square=True,           # Setiap sel = kotak sama besar
    annot_kws={'size': 11, 'fontweight': 'bold', 'color': '#1a1a1a'},
    cbar_kws={
        'label': 'Kekuatan Korelasi (r)',
        'shrink': 0.75,
        'ticks': [0, 0.2, 0.4, 0.6, 0.8, 1.0],
        'pad': 0.02
    }
)

# Label colorbar
cbar = ax.collections[0].colorbar
cbar.ax.set_ylabel('Kekuatan Korelasi (r)', fontsize=10)
cbar.ax.tick_params(labelsize=9)
cbar.ax.text(2.8, 0.05, 'Lemah',  transform=cbar.ax.transAxes, fontsize=8, color='#555', ha='left')
cbar.ax.text(2.8, 0.52, 'Sedang', transform=cbar.ax.transAxes, fontsize=8, color='#555', ha='left')
cbar.ax.text(2.8, 0.95, 'Kuat',   transform=cbar.ax.transAxes, fontsize=8, color='#555', ha='left')

# Judul
ax.set_title(
    f'Heatmap Korelasi {metode} Antar Indikator\n'
    'Biru tua = korelasi kuat  |  Putih/muda = korelasi lemah',
    fontsize=13, fontweight='bold', pad=18, color='#1a1a1a'
)

# Label sumbu — sejajar dengan kotak (ha=right + rotation=35)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=10, color='#2c2c2c')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=10, color='#2c2c2c')
ax.set_xlabel('Indikator (Sumbu X)', fontsize=11, labelpad=12, color='#2c2c2c')
ax.set_ylabel('Indikator (Sumbu Y)', fontsize=11, labelpad=10, color='#2c2c2c')
ax.tick_params(length=0)  # Hapus garis tick biar lebih bersih

# Legend di BAWAH plot (tidak bentrok dengan colorbar)
legend_items = [
    (plt.Rectangle((0,0),1,1, color='#DEEBF7'), 'r < 0.40  = Lemah'),
    (plt.Rectangle((0,0),1,1, color='#9ECAE1'), 'r 0.40-0.54 = Sedang'),
    (plt.Rectangle((0,0),1,1, color='#3182BD'), 'r 0.55-0.69 = Sedang-Kuat'),
    (plt.Rectangle((0,0),1,1, color='#08306B'), 'r >= 0.70  = Kuat'),
]
ax.legend(
    [h for h,_ in legend_items],
    [l for _,l in legend_items],
    title='Interpretasi Kekuatan r',
    title_fontsize=9,
    fontsize=9,
    loc='upper center',
    bbox_to_anchor=(0.42, -0.22),
    ncol=4,
    frameon=True,
    fancybox=True,
    framealpha=0.9,
    edgecolor='#C8963C'
)

plt.tight_layout()
plt.show()

# Print top 5 pasangan korelasi terkuat
print('=' * 62)
print(f'TOP 5 KORELASI TERKUAT ANTAR INDIKATOR ({metode.upper()})')
print('=' * 62)
corr_lower = corr_matrix.where(
    np.tril(np.ones(corr_matrix.shape), k=-1).astype(bool)
)
top5 = corr_lower.stack().sort_values(ascending=False).head(5)
for rank, ((ind1, ind2), val) in enumerate(top5.items(), 1):
    kuat = interpretasi_r(val)
    print(f'  {rank}. {ind1:<30} <-> {ind2:<30}  r = {val:.3f} ({kuat})')
print()
print(f'  Korelasi variabel utama:')
print(f'  skor_beban <-> skor_emosi : r = {r:.4f} ({kekuatan})')
print('=' * 62)


---
> ### 💡 Interpretasi — Heatmap Korelasi Antar Indikator
>
> Heatmap ini menampilkan **kekuatan hubungan antara setiap pasangan indikator**:
> - **Biru tua** (nilai mendekati 1,00) → korelasi kuat: dua indikator ini bergerak bersama.
> - **Putih/biru muda** (nilai mendekati 0,00) → korelasi lemah: dua indikator tidak berkaitan.
> - **Diagonal utama** selalu 1,00 karena setiap variabel berkorelasi sempurna dengan dirinya sendiri.
> - Baca seperti peta: **semakin gelap, semakin kuat kaitannya**.
>
> 📌 **Intinya:** Indikator-indikator emosional (mood tidak stabil, tidur memburuk, pengaruh ke emosi keseluruhan) **saling berkaitan erat satu sama lain**. Ini mempertegas bahwa tekanan akademik membentuk satu "kluster dampak" yang menyerang berbagai sisi kesehatan mental sekaligus.



## 11. Perbandingan Berdasarkan Jenis Kelamin


In [ ]:
# ======================================================
# Aku cek apakah ada perbedaan beban akademik dan kondisi
# emosional antara mahasiswa laki-laki dan perempuan.
# Aku pakai t-test untuk memvalidasi perbedaannya.
# ======================================================

gender_group = df.groupby(COL_GENDER)[['skor_beban', 'skor_emosi']].mean().round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Perbandingan Skor Berdasarkan Jenis Kelamin', fontsize=14, fontweight='bold')
warna_g = {'Perempuan': '#fd79a8', 'Laki-laki': '#74b9ff'}

for ax, (col, judul) in zip(axes, [('skor_beban', 'Beban Akademik'), ('skor_emosi', 'Dampak Emosional')]):
    vals = gender_group[col]
    colors = [warna_g.get(g, '#999') for g in vals.index]
    bars = ax.bar(vals.index, vals.values, color=colors,
                  edgecolor='white', linewidth=2, width=0.4)
    ax.set_ylim(0, 5)
    ax.axhline(y=3, color='gray', linestyle='--', alpha=0.5, label='Skor tengah (3.0)')
    ax.set_title(judul)
    ax.set_ylabel('Rata-rata Skor (1-5)')
    ax.legend(fontsize=9)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2., val + 0.08,
                f'{val}', ha='center', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

laki = df[df[COL_GENDER] == 'Laki-laki']
perempuan = df[df[COL_GENDER] == 'Perempuan']
t_b, p_b = stats.ttest_ind(laki['skor_beban'], perempuan['skor_beban'])
t_e, p_e = stats.ttest_ind(laki['skor_emosi'], perempuan['skor_emosi'])

print('=' * 55)
print('INSIGHT - Perbandingan Gender')
print('=' * 55)
print(f'  Beban Akademik:')
print(f'    Laki-laki : {laki["skor_beban"].mean():.2f}  |  Perempuan: {perempuan["skor_beban"].mean():.2f}')
print(f'    Beda statistik: {"Signifikan" if p_b < 0.05 else "Tidak signifikan"} (p={p_b:.3f})')
print()
print(f'  Dampak Emosional:')
print(f'    Laki-laki : {laki["skor_emosi"].mean():.2f}  |  Perempuan: {perempuan["skor_emosi"].mean():.2f}')
print(f'    Beda statistik: {"Signifikan" if p_e < 0.05 else "Tidak signifikan"} (p={p_e:.3f})')
print()
if p_b < 0.05 and p_e < 0.05:
    print('  --> Perbedaan antara laki-laki dan perempuan SIGNIFIKAN secara')
    print('      statistik pada kedua indikator (beban akademik & dampak emosional).')
    print('      Perempuan melaporkan skor lebih tinggi pada keduanya. Namun sampel')
    print('      masih kecil (n=50), jadi hasil ini masih bersifat eksplorasi.')
elif p_b < 0.05 or p_e < 0.05:
    print('  --> Perbedaan gender signifikan secara statistik hanya pada salah satu')
    print('      indikator. Dengan sampel 50 orang, hasil ini perlu dikonfirmasi lagi')
    print('      dengan sampel yang lebih besar.')
else:
    print('  --> Dengan sampel 50 orang, perbedaan antar gender belum cukup besar')
    print('      untuk signifikan secara statistik.')


---
> ### 💡 Interpretasi — Perbandingan Berdasarkan Jenis Kelamin
>
> Grafik ini membandingkan rata-rata skor **beban akademik** dan **dampak emosional** antara mahasiswa laki-laki dan perempuan:
>
> - Skor **perempuan lebih tinggi** → perempuan cenderung merasakan beban/dampak emosional lebih besar.
> - **Uji t-test** memastikan apakah perbedaan ini nyata atau hanya variasi kebetulan:
>   - **p < 0,05** → perbedaan nyata secara statistik ✅
>   - **p ≥ 0,05** → perbedaan **tidak signifikan** (bisa jadi kebetulan dari sampel kecil).
>
> 📌 **Intinya:** Dengan hanya 50 responden, perbedaan antara gender kemungkinan besar **tidak signifikan secara statistik**. Diperlukan sampel yang lebih besar untuk membuktikan apakah gender benar-benar memengaruhi tingkat beban dan emosi mahasiswa.



## 12. Kesimpulan & Rekomendasi


In [ ]:
# ======================================================
# Bagian penutup - aku rangkum semua temuan penting
# dari analisis yang udah aku lakuin.
# ======================================================

# Pakai Spearman (sesuai hasil uji normalitas — data beban tidak normal)
r_val, p_val = stats.spearmanr(df['skor_beban'], df['skor_emosi'])
metode_akhir = 'Spearman'
skor_b = df['skor_beban'].mean().round(2)
skor_e = df['skor_emosi'].mean().round(2)
kekuatan_final = interpretasi_r(r_val)
top_faktor_list = pd.Series(all_faktor).value_counts().head(3).index.tolist()
kat_b = 'TINGGI' if skor_b >= 3.5 else ('SEDANG' if skor_b >= 2.5 else 'RENDAH')
kat_e = 'CUKUP SIGNIFIKAN' if skor_e >= 3.0 else 'RENDAH'

print('=' * 62)
print('KESIMPULAN PENELITIAN')
print('=' * 62)
print()
print('Judul  : Analisis Korelasi Beban Akademik terhadap')
print('         Stabilitas Emosional Mahasiswa')
print('Sampel : 50 responden dari berbagai program studi')
print()
print('TEMUAN UTAMA:')
print()
print(f'1. BEBAN AKADEMIK (rata-rata: {skor_b}/5 --> {kat_b})')
print('   Faktor utama yang dirasakan paling berat:')
for i, f in enumerate(top_faktor_list, 1):
    print(f'     {i}. {f}')
print()
print(f'2. DAMPAK EMOSIONAL (rata-rata: {skor_e}/5 --> {kat_e})')
print('   Gangguan terbesar: kualitas tidur memburuk & mood tidak stabil')
print()
print(f'3. KORELASI {metode_akhir} (r = {r_val:.4f}, p = {p_val:.4f})')
print(f'   --> Terdapat korelasi POSITIF yang {kekuatan_final.upper()}')
print(f'       antara beban akademik dan gangguan emosional.')
if p_val < 0.05:
    print('   --> Hubungan ini SIGNIFIKAN secara statistik (p < 0.05).')
    print('       Bukan kebetulan -- ada pola nyata di data ini.')
print()
print('REKOMENDASI:')
print()
print('Untuk Mahasiswa:')
print('  - Gunakan teknik manajemen waktu seperti time-blocking')
print('  - Jangan ragu minta bantuan teman atau dosen')
print('  - Istirahat itu produktif, bukan malas!')
print()
print('Untuk Institusi:')
print('  - Pertimbangkan distribusi deadline yang lebih merata')
print('  - Sediakan layanan konseling akademik yang mudah diakses')
print('  - Evaluasi beban SKS dan kompleksitas materi per semester')
print()
print('=' * 62)
